# A/B testing in R (draft notes)

## A/B testing workflow

<br>(i) **Define the objective**
   - Specify the business goal (e.g., conversion, retention)
   - Choose primary and secondary metrics
   - Determine the minimum detectable effect (MDE)

<br>(ii) **Design the experiment**
   - Define treatment and control conditions
   - Specify inclusion criteria and sampling strategy
   - Perform power analysis to determine sample size

<br>(iii) **Randomisation**
   - Randomly sample users from the target population
   - Randomly assign users to control and treatment groups

<br>(iv) **Run and monitor**
   - Ensure data integrity and experiment validity
   - Check for sample ratio mismatch (SRM)

<br>(v) **Statistical inference**
   - Estimate treatment effects
   - Test for statistical significance
   - Report confidence intervals and effect sizes

## Guide to the toolkit

Comparing groups
- parametric
   * t-test (`t.test()`)
- non-parametric
   * Mann-Whitney U test (`wilcox.test()`)
   * Chi-squared test (`chisq.test()`)
   * Fisher's exact test (`fisher.test()`)

Associating variables
- parametric
   * Pearson correlation (`cor.test()`)
- non-parametric
   * Spearman correlation (`cor.test()`)

Predicting data
- parametric
   * linear regresion (`lm()`)
- non-parametric
   * logit (`glm()`)

## Visuals

| hypothesis | test | graphic |
| :-: | :-: | :-: |
| comparing means | t-test | bar plots with error bars for std dev, std error or confidence intervals |
| comparing medians | Mann-Whitney U, Chi squared, Fisher's exact, t-test | box plots |
| correlations | correlation coefficients | scatterplots with a trend, direction and rate |

## Toolkit

1. Statistical framework
2. Independent sample t-test
3. Proportion test
4. Mann-Whitney U test
5. Chi squared test
6. Fisher's exact test
7. p-values to scores and vice versa in R
8. Correlational analysis
9. Linear regression

## 1 Statistical framework

### Confusion matrix

| | predicted positive class | predicted negative class | |
| :-: | :-: | :-: | :-: |
| **actual positive class** | True positive<br>$(1-\beta)$ | False negative<br>$(\beta)$ | Sensitivity<br>$\left(\frac{TP}{TP+FN}\right)$ |
| **actual negative class** | False positive<br>$(\alpha)$ | True negative<br>$(1-\alpha)$ | Specificity<br>$\left(\frac{TN}{TN+FP}\right)$ |
| | Precision<br>$\left(\frac{TP}{TP+FP}\right)$ | Negative predictive value<br>$\left(\frac{TN}{TN+FN}\right)$ | Accuracy<br>$\left(\frac{TP+TN}{TP+TN+FP+FN}\right)$ |

### Family-wise error rate

Family-wise error rate captures the idea that as the number of hypothesis tests increases, so does the probability of observing at least one false positive. FWER is the probability of having at least one false positive across all tests, which grows with the number of tested variables, number of tests or fluctuation in the sample.

$$
FWER = 1 - (1-\alpha)^m
$$

Where 
- $\alpha$ denotes the probability of Type I Error (false positive),
- $m$ is the number of tests. 

## 2 Independent sample t-test

The t-test is used when we are interested in the mean outcome.

### t-test

An independent sample t-test assumes
- a random sample
- normally distributed data
- similar group variances

Assess assumptions
- Levene's test for variances
- Shapiro-Wilk or Jarque-Bera for normality
- histograms and QQ plots for the shape of the data

For a general hypothesis test $H_0: \theta = \theta_0$ under standard regularity conditions $W \overset{d}{\to}\mathcal{N}(0,1)$,

$$
W =\frac{\hat{\theta}-\theta_0}{SE(\hat{\theta})}, \qquad SE(\hat{\theta}) = \sqrt{\widehat{Var}(\hat{\theta})}
$$

#### Summary table

| test | statistic | distribution under null | confidence interval |
| :--: | :-------: | :----------: | :-----------------: |
| one sample t-test | $t=\frac{\bar{x}-\mu_0}{s/\sqrt{n}}$ | $$\begin{align*}t&\sim t_{df}, \\ \quad df&=n-1 \end{align*}$$ | $\bar{x}\pm t_{\alpha/2, df} \times \frac{s}{\sqrt{n}}$ |
| two-sample t-test<br>(equal variances) | $$\begin{align*} t&=\frac{\bar{x}_1 - \bar{x}_2}{s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}}, \\ \quad s_p^2&=\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2} \end{align*}$$ | $$\begin{align*} t&\sim t_{df}, \\ \quad df&=n_1+n_2-2 \end{align*}$$ | $(\bar{x}_1-\bar{x}_2)\pm t_{\alpha/2, df} \times s_p\sqrt{\frac{1}{n_1}+\frac{1}{n_2}}$ |
| two-sample t-test<br>(unequal variances) | $t=\frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}}$ | $$\begin{align*} t&\sim t_{df}, \\ \quad df&=\frac{\left(\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}\right)^2}{\frac{(s_1^2/n_1)^2}{n_1-1}+\frac{(s_2^2/n_2)^2}{n_2-1}} \end{align*}$$ | $(\bar{x}_1-\bar{x}_2)\pm t_{\alpha/2, df} \times \sqrt{\frac{s_1^2}{n_1}+\frac{s_2^2}{n_2}}$ |
| one sample z-test | $z=\frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}$ | $z\sim \mathcal{N}(0,1)$ | $\bar{x}\pm z_{\alpha/2} \times \frac{\sigma}{\sqrt{n}}$ |
| two-sample z-test | $z=\frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{\sigma_1^2}{n_1}+\frac{\sigma_2^2}{n_2}}}$ | $z\sim \mathcal{N}(0,1)$ | $(\bar{x}_1-\bar{x}_2)\pm z_{\alpha/2} \times \sqrt{\frac{\sigma_1^2}{n_1}+\frac{\sigma_2^2}{n_2}}$ |
| paired t-test | $$\begin{align*} t&=\frac{\bar{d}-0}{s_d/\sqrt{n}}, \\ \quad d_i&=x_{1i}-x_{2i} \end{align*}$$ | $$\begin{align*}t&\sim t_{df}, \\ \quad df&=n-1\end{align*}$$ | $\bar{d}\pm t_{\alpha/2, df} \times \frac{s_d}{\sqrt{n}}$ |

Recall that for $X\perp Y$, $Var(X-Y)=Var(X)+Var(Y)=Var(X+Y)$.

### Null hypothesis

One-sample:

$$\begin{align*}
H_0: \mu = \mu_0
\end{align*}$$

Two-sample:

$$\begin{align*}
H_0: \mu_1 &= \mu_2 \\
\mathbb{E}[Y \mid \text{group} = A] &= \mathbb{E}[Y \mid \text{group} = B]
\end{align*}$$

This is a test of no difference between the population means.

#### Implementation in R

``` R
# Test normality with Shapiro-Wilk
# H0: X ~ N
stats::shapiro.test(x)

# Test normality with Jarque-Bera
# H0: X ~ N
tseries::jarque.bera.test(x)

# Assess group variances
# H0: var(x1) = var(x2)
car::leveneTest(x1 ~ x2, data = df)

# Two-sample t-test 
# H0: E[Y | group = A] = E[Y | group = B]
stats::t.test(
  metric ~ group,
  data = df,
  alternative = "two.sided",
  conf.level = 0.95,
  var.equal = FALSE
)
```

### Cohen's d

Effect size.

$$d = \frac{\bar{x}_1-\bar{x}_2}{s_{\text{pooled}}}$$

Where
- $s_{\text{pooled}}=\sqrt{\frac{s_1^2+s_2^2}{2}}$ for equal sample sizes,
- $s_{\text{pooled}}=\sqrt{\frac{(n_1-1)s_1^2+(n_2-1)s_2^2}{n_1+n_2-2}}$ for unequal sample sizes.

One-sample effect size uses an assumed mean, $d=\frac{\bar{x}-\mu}{s}$.

| d | effect size |
| :-: | :-: |
| $\approx .2$ | small |
| $\approx .5$ | medium |
| $\approx .8$ | large | 

#### Implementation in R

``` R
# Effect size for the two-sample t-test
# Target: standardized mean difference
# H0 (test): E[Y | A] = E[Y | B]
effectsize::cohens_d(
  metric ~ group,
  data = df,
  pooled_sd = FALSE,
  ci = 0.95
)
```

### Power

Power analysis allows us to compute the unknown, such as the required sample size.

To derive the formula, start with two [CLT](https://www.probabilitycourse.com/chapter7/7_1_2_central_limit_theorem.php) equations $Z_{\alpha/2}=\frac{\bar{x}-\mu_0}{\sigma/\sqrt{n}}$ and -$Z_{\beta}=\frac{\bar{x}-\mu_0+\delta}{\sigma/\sqrt{n}}$. Solve for $\bar{x}$, equate the expressions and solve for n, eliminating $\mu_0$ in the process.

$$\begin{align*}
n &= \frac{2\sigma^2 (Z_{\alpha/2} + Z_\beta)^2}{\delta^2} \\
&= \frac{2(Z_{\alpha/2} + Z_\beta)^2}{\left(\frac{\delta}{\sigma}\right)^2} \\
&= \frac{2(Z_{\alpha/2} + Z_\beta)^2}{d^2}
\end{align*}$$

Where 
- $\sigma$ is the unknown population standard deviation,
- $\alpha$ is the probability of a false positive (typically 0.05),
- $Z_{\alpha/2}$ the corresponding critical value for a two-tailed test (typically 1.96),
- $\beta$ is the probability of a false negative (typically 0.2),
- $Z_\beta$ the corresponding critical value (typically 0.84),
- $\delta$ is the effect size,
- $d$ the effect size in standard deviations (Cohen's d).

#### Implementation in R

``` R
# Power analysis (NULL as required)
# Equal sample sizes
pwr::pwr.t.test(
  n = NULL,
  d = 0.2, # Standardized mean difference = (delta = mu_B − mu_A) / sigma
  power = 0.8,
  sig.level = 0.05,
  type = "two.sample",
  alternative = "two.sided"
)

# Unequal sample sizes
pwr::pwr.t2n.test(
  n1 = NULL,
  n2 = 200,
  d = 0.2,
  sig.level = 0.05,
  power = 0.8,
  alternative = "two.sided"
)
```

## 3 Proportions test

Use proportions when the KPI is a binary
- conversion
- signup
- churn
- click (or not)
- purchase (or not)

In large samples, the proportion and the t-test tend towards the same quantity.

### Structure of binary data

Table the success counts,

|   | success | failure | group margin |
| :-: | :-: | :-: | :-: | 
| **treatment** | $n_{11}$ | $n_{12}$ | $n_t=n_{11}+n_{12}$ |
| **control** | $n_{21}$ | $n_{22}$ | $n_c=n_{21}+n_{22}$ |
| **outcome margin** | $n_s=n_{11}+n_{21}$ | $n_f=n_{12}+n_{22}$ | $n=n_t+n_c=n_s+n_f$ |

Compute the proportions,

|   | success | failure | group margin |
| :-: | :-: | :-: | :-: | 
| **treatment** | $\hat{p}_1=\frac{n_{11}}{n_t}$ | $1-\hat{p}_1=\frac{n_{12}}{n_t}$ | $\hat{p}_t=1$ |
| **control** | $\hat{p}_2=\frac{n_{21}}{n_c}$ | $1-\hat{p}_2=\frac{n_{22}}{n_c}$ | $\hat{p}_c=1$ |

In terms of probabilities,

|   | success ($Y=1$) | failure ($Y=0$) | marginal $X$ |
| :-: | :-: | :-: | :-: |
| **treatment** ($X=1$) | $p_1=P(Y = 1 \mid X = 1)$ | $(1-p_1)=P(Y = 0 \mid X = 1)$ | $P(X=1)=1$ |
| **control** ($X=0$) | $p_2=P(Y = 1 \mid X = 0)$ | $(1-p_2)=P(Y = 0 \mid X = 0)$ | $P(X=0)=1$ |

### z-test for proportions

Two-sample z-test compares proportions between groups. It assumes
- large sample ($np \geq 5$, $n(1-p) \geq 5$), 
- independent observations.

#### Summary table

| test | statistic | distribution | confidence interval |
| :--: | :-------: | :----------: | :-----------------: |
| two-sample proportion test | $z=\frac{\hat p_1-\hat p_2}{\sqrt{\hat p_{pooled}(1-\hat p_{pooled})(\frac{1}{n_t}+\frac{1}{n_c})}}$ | $z\sim \mathcal{N}(0,1)$ | $(\hat p_1-\hat p_2)\pm z_{\alpha/2}\sqrt{\frac{\hat p_1(1-\hat p_1)}{n_t}+\frac{\hat p_2(1-\hat p_2)}{n_c}}$ |

### Null hypothesis

$$H_0: p_1=p_2$$

This is a test of no difference between the proportions.

#### Implementation in R

``` R
# Two-sample proportion test
# H0: p_1 = p_2
tab <- with(df, table(group, metric))
#tab <- with(df, table(group, as.integer(metric))) # For logical binaries

stats::prop.test(
  x = c(tab["A", "1"], tab["B", "1"]),
  n = c(sum(tab["A", ]), sum(tab["B", ])),
  alternative = "two.sided",
  conf.level = 0.95,
  correct = FALSE # Yates continuity correction (reduces power)
)
```

### Cohen's h

Effect size for proportions.

$$h = 2\arcsin{(\sqrt{p_1})} - 2\arcsin{(\sqrt{p_2})}$$

| h | effect size |
| :-: | :-: |
| $\approx .2$ | small |
| $\approx .5$ | medium |
| $\approx .8$ | large |

### Risk difference, risk ratio and odds ratio

Risk difference $\delta$ captures the absolute difference. Risk ratio $\rho$ captures the multiplicative effect on the probability scale. Odds ratio $\theta$ captures the multiplicative effect on the odds scale.

#### Risk difference

**Risk difference** is a measure of the effect size in terms of the difference between the groups. In brief, 
$$
\begin{align*}
\text{risk difference} &= \text{risk in treatment} - \text{risk in control} \\
\delta &= p_A-p_B \\
\Rightarrow \hat{\delta} &= \hat{p}_A-\hat{p}_B
\end{align*}
$$

Assume:

- $H_0: \delta = 0$ (i.e. there is no difference between the groups)
- $H_a: \delta \neq 0$ (i.e. this is a two-tailed test where the difference between the groups can be either larger or smaller than 0)
- $\alpha=0.05 \Rightarrow z_{1-\alpha/2} \approx 1.96$ (i.e. 5% of the probability mass falls into the rejection region and this corresponds to the tails 1.96 standard deviations away either side of the mean)

Begin with computing the test statistic, $$z = \frac{\hat{p}_A-\hat{p}_B-\delta}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_t}+\frac{1}{n_c}\right)}}, \quad
\hat{p} = \frac{n_{11}}{n_t}+\frac{n_{21}}{n_c}$$

**Decision rule**: We reject $H_0$ if $|z| > z_{1-\alpha/2}$. In this case, the result is said to be statistically significant.

Next, construct the confidence interval, $$\delta \in \hat{p}_A-\hat{p}_B \pm Z_{1-\alpha} \text{SE}\left(\hat{\delta}\right), \quad
\text{SE}\left(\hat{\delta}\right) = \sqrt{\frac{\hat{p}_A(1-\hat{p}_A)}{n_t} + \frac{\hat{p}_B(1-\hat{p}_B)}{n_c}}$$

**Decision rule**: We reject $H_0$ if $\delta \notin CI_{\delta}$.

#### Risk ratio

**Risk ratio** measures the effect size as a ratio of the same quantities. In brief,
$$
\begin{align*}
\text{risk ratio} &= \frac{\text{risk in treatment}}{\text{risk in control}} \\
\rho &= \frac{p_A}{p_B} \\
\Rightarrow \hat{\rho} &= \frac{\hat{p}_A}{\hat{p}_B}
\end{align*}
$$

Assume:

- $H_0: \rho = 1$ (i.e. there is no difference between the groups)
- $H_a: \rho \neq 1$ (i.e. this is a two-tailed test where the difference between the groups can be either larger or smaller than 0)
- $\alpha=0.05 \Rightarrow z_{1-\alpha/2} \approx 1.96$

Construct the confidence interval,

$$
\rho \in \exp{\left( \ln{\hat{\rho}}\pm Z_{1-\alpha/2}\sqrt{\widehat{\text{SE}}(\ln{\hat{\rho}})} \right)}, \quad
\widehat{\text{SE}}(\log \hat \rho)
= \sqrt{\frac{1 - \hat p_A}{\hat p_A n_t} + \frac{1 - \hat p_B}{\hat p_B n_c}}=\sqrt{\frac{1}{n_{11}} - \frac{1}{n_{11}+n_{12}} + \frac{1}{n_{21}} - \frac{1}{n_{21}+n_{22}}}
$$

Note: Look up the **Haldane-Anscombe correction** if any of the data counts is zero.

**Decision rule**: We reject $H_0$ if $\rho \notin CI_{\rho}$.
#### Odds ratio

**Odds ratio** measures the effect size as the ratio of the odds. In brief,
$$
\begin{align*}
\text{odds ratio} &= \frac{\text{odds in treatment}}{\text{odds in control}} \\
\theta &= \frac{p_A/(1-p_A)}{p_B/(1-p_B)} \\
\Rightarrow \hat{\theta} &= \frac{\hat{p}_A/(1-\hat{p}_A)}{\hat{p}_B/(1-\hat{p}_B)}
\end{align*}
$$

Assume:

- $H_0: \theta = 1$ (i.e. there is no difference between the groups)
- $H_a: \theta \neq 1$ (i.e. this is a two-tailed test where the difference between the groups can be either larger or smaller than 0)
- $\alpha=0.05 \Rightarrow z_{1-\alpha/2} \approx 1.96$

Construct the confidence interval,

$$
\theta \in \exp{\left( \ln{ \hat \theta \pm Z_{1-\alpha/2}\ \widehat{\mathrm{SE}}(\ln{\hat{\theta}}) } \right)}, \quad
\widehat{\text{SE}}(\ln{\hat{\theta}})
= \sqrt{\frac{1}{n_{11}}+\frac{1}{n_{12}}+\frac{1}{n_{21}}+\frac{1}{n_{22}}}.
$$

Note: Look up the **Haldane-Anscombe correction** if any of the data counts is zero.

**Decision rule**: We reject $H_0$ if $\theta \notin CI_{\theta}$.

#### Implementation in R

``` R
# Effect size for the two-sample proportions test
# Target: difference in proportions/risk ratio/odds ratio
# H0 (test): p_A = p_B
effectsize::difference_in_proportions(
  metric ~ group,
  data = df,
  ci = 0.95
)

effectsize::oddsratio(
  metric ~ group,
  data = df,
  ci = 0.95
)

effectsize::riskratio(
  metric ~ group,
  data = df,
  ci = 0.95
)
```

### Power analysis

``` R
# Power analysis (NULL as required)
# Equal sample sizes
pwr::pwr.2p.test(
  n = NULL,
  h = pwr::ES.h(p1 = 0.10, p2 = 0.12), # Converts delta_p = p_B - p_A to Cohen's h
  power = 0.8,
  sig.level = 0.05,
  alternative = "two.sided"
)

# Unequal sample sizes
pwr::pwr.2p2n.test(
    n1 = NULL,
    n2 = 200,
    h = .8, 
    power = .8, 
    sig.level = .05,
    alternative = "two.sided"
)
```

## 4 Non-parametric test

Use the Mann-Whitney rank-sum test for non-normal data. In a nutshell, we are comparing the shapes of the distributions where the null assumes both groups have the same shape.

Typical use
- the outcome is ordinal
- the metric is very heavy-tailed
- a rank-based comparison (as opposed to focus on mean uplift)

### Mann-Whitney U test

For two groups $g \in \{A,B\}$, we compute $U_A$ and $U_B$,

$$\begin{align*}
U_A &= n_An_B+\frac{n_A(n_A+1)}{2}-T_A \\ 
U_B &= n_An_B+\frac{n_B(n_B+1)}{2}-T_B \\
U&=\text{min}(U_A,U_B)
\end{align*}$$

Where $T_g$ denotes the rank sum for group $g$, $n_g$ the number of cases in group $g$.

We can express the rank-sum formula alternatively,

$$
U=\sum_{i=1}^{n_B}\sum_{j=1}^{n_A}
\left[I_{\{Y_{Bi}>Y_{Aj}\}}+\frac12 I_{\{Y_{Bi}=Y_{Aj}\}}\right]
$$

Expected value $\mu_U$, standard error $\sigma_U$ and the z-score $z$:

$$\begin{align*}
\mu_U&=\frac{n_An_B}{2} \\
\sigma_U&=\sqrt{\frac{n_An_B(n_A+n_B+1)}{12}} \\
z&=\frac{U-\mu_U}{\sigma_U}
\end{align*}$$

### Null hypothesis

$$
H_0: P(Y_A > Y_B) + 0.5 P(Y_A = Y_B) = 0.5
$$

$Y_A$ is a randomly drawn observation from group $A$, $Y_B$ from group $B$.

This is a test of no difference between distributions.

#### Implementation in R

``` R
# Mann-Whitney / Wilcoxon rank-sum test
# H0: P(Y_A > Y_B) + 0.5 P(Y_A = Y_B) = 0.5
stats::wilcox.test(
  metric ~ group,
  data = df,
  alternative = "two.sided",
  conf.int = TRUE,
  conf.level = 0.95,
  exact = FALSE # Exact or normal approximation
)
```

### Rank biserial correlation

Effect size for non-parametric tests. $r_{rb}\in[-1,1]$.

$$
r_{rb}=2\left[\Pr(Y_B>Y_A)+\frac12\Pr(Y_B=Y_A)\right]-1
$$

#### Implementation in R

``` R
# Effect size for the Mann-Whitney / Wilcoxon rank-sum test
# Target: rank-biserial correlation / probability-of-superiority scale
# H0 (test): P(Y_B > Y_A) + 0.5 P(Y_B = Y_A) = 0.5
effectsize::rank_biserial(
  metric ~ group,
  data = df,
  ci = 0.95
)
```

### Power analysis

``` R
# Target: theta = P(Y_B > Y_A) + 0.5 P(Y_B = Y_A)
MKpower::sim.ssize.wilcox.test(
  n1 = NULL,
  n2 = 200,
  alpha = 0.05,
  power = 0.8,
  alternative = "two.sided",
  distribution = "normal", # Or specify custom distributions
  location.shift = 0.5 # Effect encoded as shift
)
```

## 5 Chi squared test of independence

The Chi squared test tests the relationship between two categorical variables.

Let us assume variable 1 has $r$ categories and variable 2 has $c$ categories. Construct a contingency (frequency) table with cell values,

$$O_{ij}$$

Where $i=1,\dots,r$,$j=1,\dots,c$ and $\sum_{i=1}^r \sum_{j=1}^c O_{ij} = n$. Note that the test assumes $E_{ij} \geq 5$ for all cells.

Then, the row totals and column totals are,

$$
O_{i\cdot} = \sum_{j=1}^c O_{ij}, \qquad
O_{\cdot j} = \sum_{i=1}^r O_{ij}
$$

Now we can compute the expected cell probability $p_{ij}^{(0)}$ and the expected cell count is $E_{ij}$ under the null,

$$\begin{align*}
P(\text{row } i,\text{ col } j) &= P(\text{row } i)\,P(\text{col } j) \\
p_{ij}^{(0)} &= \left(\frac{O_{i\cdot}}{n}\right)\left(\frac{O_{\cdot j}}{n}\right) \\
E_{ij} &= n p_{ij}^{(0)} = \frac{O_{i\cdot} O_{\cdot j}}{n}
\end{align*}$$

Chi squared statistic,

$$\begin{align*}
X^2&=\sum_{i=1}^r \sum_{j=1}^c\frac{(O_{ij} - E_{ij})^2}{E_{ij}} \\
&=\sum_{i=1}^r \sum_{j=1}^c
\frac{O_{ij}^2}{E_{ij}} - n
\end{align*}$$

Degrees of freedom:

$$df = \left(r - 1\right) \times \left(c - 1\right)$$

And the test statistic follows the Chi squared distribution,

$$
X^2 \sim \chi_{df}^2
$$

### Null hypothesis

$$
H_0: p_{ij} = p_{ij}^{(0)} \quad \forall\, i,j
$$

Where $p_{ij} = \frac{O_{ij}}{n}$ are observed cell probabilities and $p_{ij}^{(0)} = \frac{O_{i\cdot}}{n}\frac{O_{\cdot j}}{n}$ are expected cell probabilities under independence.

This is a test of no dependence between the variables.

#### Implementation in R

``` R
# Chi squared statistic
# H0: no relationship (independent variables)
stats::chisq.test(
	table(
		df$var1,
		df$var2
	)
)
```

### Cohen's w

Effect size.

$$\begin{align*}
w&=\sqrt{
\sum_{i=1}^r \sum_{j=1}^c
\frac{\left(p_{ij} - p_{ij}^{(0)}\right)^2}{p_{ij}^{(0)}}
} \\
&= \sqrt{\frac{X^2}{n}}
\end{align*}$$

| w | effect size |
| :-: | :-: |
| $\approx .1$ | small |
| $\approx .3$ | medium |
| $\approx .5$ | large |

#### Implementation in R

``` R
# Compute Cohen's w
freq_tab <- table(df$var1, df$var2)
pwr::ES.w2(
	freq_tab / sum(freq_tab)
)
```

### Power analysis

``` R
# Power analysis (NULL as required)
pwr::pwr.chisq.test(
  w = 0.1,
  df = 2,
  N = NULL,
  power = 0.8,
  sig.level = 0.05
)
```

## 6 Fisher's exact test

Fisher's test is small-sample alternative to the Chi squared test. It computes the p-value from the cumulative probability in the tail(s) of the hypergeometric distribution directly.

Assume a 2 by 2 table,

| | col 1 | col 2 | row total |
| :-: | :-: | :-: | :-: |
| **row 1** | $n_{11}$ | $n_{12}$ | $n_1 = n_{11}+n_{12}$ |
| **row 2** | $n_{21}$ | $n_{22}$ | $n_2 = n_{21}+n_{22}$ |
| **col total** | $m_1=n_{11}+n_{21}$ | $m_2=n_{12}+n_{22}$ | $N$ |

The proportions are then $p_A = n_{11}/n_1, p_B = n_{21}/n_2$.

### Test statistic (hypergeometric model)

Condition on fixed margins $(n_1, n_2, m_1)$. Then

$$n_{11} \sim \text{Hypergeometric}(N, m_1, n_1)$$

The probability of observing a table with value $j$ in the $(1,1)$ follows.

Right-tailed p-value,

$$
p = \sum_{j=n_{11}}^{\min(n_1,\, m_1)} 
\frac{\binom{n_1}{j} \binom{n_2}{m_1-j}}{\binom{N}{m_1}}
$$

Two-tailed p-value,

$$
p = \sum_{\substack{j=0 \\ P(j) \leq P(n_{11})}}^{\min(n_1,\, m_1)} 
\frac{\binom{n_1}{j} \binom{n_2}{m_1-j}}{\binom{N}{m_1}}
$$

Where $P(n_{11})$ is the probability of the observed table and 
$P(j) = \frac{\binom{n_1}{j} \binom{n_2}{m_1-j}}{\binom{N}{m_1}}$.

### Null hypothesis

$$H_0: p_A = p_B$$

Where $p_A = n_{11}/n_1$ and $p_B = n_{21}/n_2$. Equivalently, $H_0: \text{OR} = 1$ (equal odds).

This is a test of no difference between the proportions (independence).

#### Implementation in R

``` R
# Fisher's exact test for count data
# H0: p_1 = p_2 (variables are independent)
freq_tab <- table(df$var1, df$var2)
stats::fisher.test(freq_tab)
```

### Odds ratio

Effect size.

$$\begin{align*}
\text{OR}&=\frac{n_{11} n_{22}}{n_{12} n_{21}} \\
&=\frac{p_A/(1-p_A)}{p_B/(1-p_B)}
\end{align*}$$

| odds ratio | effect size |
| :-: | :-: |
| $\approx 1$ | no effect |
| $\approx 1.5$ | small |
| $\approx 2.5$ | medium |
| $\approx 4$ | large |

#### Implementation in R

``` R
# Effect size
effectsize::oddsratio(
  metric ~ group,
  data = df,
  ci = 0.95
)
```

### Cohen's h

$$\begin{align*}
h&=2\left[\arcsin(\sqrt{p_A}) - \arcsin(\sqrt{p_B})\right] \\
&\approx \sqrt{p(1-p)} \cdot \log(\text{OR})
\end{align*}$$

#### Implementation in R

``` R
# Power analysis (NULL as required)
pwr::pwr.2p.test(
  h = pwr::ES.h(p1 = 0.10, p2 = 0.15), # Cohen's h
  n = NULL,
  power = 0.8,
  sig.level = 0.05
)
```

## 7 Critical value and p-value helpers

``` R
# Two-tailed p-value from t-statistic
p_from_t <- function(t_value, df, tails = "two") {
    if (tails == "two") {
        return(2 * pt(-abs(t_value), df))
    } else if (tails == "lower") {
        return(pt(t_value, df, lower.tail = TRUE))
    } else if (tails == "upper") {
        return(pt(t_value, df, lower.tail = FALSE))
    }
}

# Get t-statistic from p-value
t_from_p <- function(p_value, df, tails = "two") {
    if (tails == "two") {
        return(qt(1 - p_value / 2, df))
    } else if (tails == "lower") {
        return(qt(p_value, df))
    } else if (tails == "upper") {
        return(qt(1 - p_value, df))
    }
}

# Two-tailed p-value from z-statistic
p_from_z <- function(z_value, tails = "two") {
    if (tails == "two") {
        return(2 * pnorm(-abs(z_value)))
    } else if (tails == "lower") {
        return(pnorm(z_value, lower.tail = TRUE))
    } else if (tails == "upper") {
        return(pnorm(z_value, lower.tail = FALSE))
    }
}

# Get z-statistic from p-value
z_from_p <- function(p_value, tails = "two") {
    if (tails == "two") {
        return(qnorm(1 - p_value / 2))
    } else if (tails == "lower") {
        return(qnorm(p_value))
    } else if (tails == "upper") {
        return(qnorm(1 - p_value))
    }
}
```

## 8 Correlational analysis

Correlation coefficient $r\in[-1,1]$ measure the degree of association between two variables.

Let the paired observations be

$$
(X_1, Y_1), \ldots, (X_n, Y_n).
$$

Common correlation measures are,

8.1 **Pearson correlation**: linear association between two numerical variables  
8.2 **Spearman correlation**: monotonic association based on ranks  
8.3 **Kendall correlation**: ordinal association based on concordant and discordant pairs

| coefficient | formula | concept |
| :-: | :-: | :-: | 
| Pearson | $\rho = \frac{\operatorname{Cov}(X,Y)}{\sigma_X \sigma_Y}$ | Pearson uses the raw numerical values. |
| Spearman | $\rho_s = \operatorname{Corr}(\operatorname{rank}(X), \operatorname{rank}(Y))$ | Spearman uses the ranks. |
| Kendall | $\tau = P(\text{concordant}) - P(\text{discordant})$ | Kendall has a probability interpretation based on pairwise ordering. |

### 8.1 Pearson correlation

Pearson's correlation coefficient measures the strength of **linear association between** two numerical variables.

Use Pearson correlation when,
- both variables are numerical
- the association is approximately linear
- outliers are not dominating the relationship
- you care specifically about linear dependence

Assumptions
- use the Shapiro-Wilk test to assess the normality of each variable

#### Population parameter

$$
\rho
=\frac{\operatorname{Cov}(X,Y)}{\sigma_X \sigma_Y}
=\frac{\mathbb{E}[(X-\mu_X)(Y-\mu_Y)]}{\sigma_X \sigma_Y}.
$$

#### Sample statistic

$$
r
=\frac{\sum_{i=1}^n (x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum_{i=1}^n (x_i-\bar{x})^2}\sqrt{\sum_{i=1}^n (y_i-\bar{y})^2}}.
$$

#### Null hypothesis

$$
H_0: \rho = 0
$$

This is a test of no linear association.

#### Test statistic

Under the usual assumptions,

$$
t = \frac{r\sqrt{n-2}}{\sqrt{1-r^2}} \sim t_{n-2}
\quad \text{under } H_0
$$

##### Implementation in R

``` R
# Pearson correlation between x and y
# H0: rho = 0
stats::cor.test(
  x = df$x,
  y = df$y,
  method = "pearson",
  alternative = "two.sided"
)

# Group-blind correlation between x and y
stats::cor.test(
	~ x + y,
	data = df,
	method = "pearson"
)

# Correlation between x and y in a subset of z
stats::cor.test(
	~ x + y,
	data = df,
	subset(z == "val"),
	method = "pearson"
)
```

#### Effect size

The correlation coefficient itself is the effect size. Conventionally,

| $\vert r\vert$ | effect size |
| :-: | :-: |
| $\approx 0.1$ | small |
| $\approx 0.3$ | medium |
| $\approx 0.5$ | large |

#### Power analysis

``` R
# Power analysis (NULL as required)
pwr:pwr.r.test(
	n = NULL,
	r = 0.3,
	power = 0.8,
	sig.level = 0.05
)
```

### 8.2 Spearman correlation

Spearman's correlation coefficient measures the strength of a **monotonic association** between two variables.

Use Spearman correlation when:

- the relationship is monotonic but not necessarily linear
- the variables are ordinal or not well behaved numerically
- outliers make Pearson unreliable
- you want a rank-based measure of association

Assumptions
- a monotonic relationship between data points

It is the Pearson correlation computed on the **ranks** of the data.

Let

$$
R_i = \operatorname{rank}(X_i), \qquad S_i = \operatorname{rank}(Y_i).
$$

Then Spearman's rank correlation is

$$
\rho_s = \operatorname{Corr}(R,S).
$$

#### Sample statistic

$$
r_s =
\frac{\sum_{i=1}^n (R_i-\bar{R})(S_i-\bar{S})}
{\sqrt{\sum_{i=1}^n (R_i-\bar{R})^2}\sqrt{\sum_{i=1}^n (S_i-\bar{S})^2}}.
$$

If there are no ties, an equivalent formula is

$$
r_s = 1 - \frac{6\sum_{i=1}^n d_i^2}{n(n^2-1)},
$$

where

$$
d_i = R_i - S_i.
$$

#### Null hypothesis

$$
H_0: \rho_s = 0
$$

This is a test of no monotonic association.

#### Implementation in R

``` R
# Spearman correlation
# H0: rho_s = 0
stats::cor.test(
  x = df$x,
  y = df$y,
  method = "spearman",
  alternative = "two.sided",
  exact = FALSE
)

stats::cor.test(
	~ x + y,
	data = df,
	method = "spearman",
	exact = FALSE
)

stats::cor.test(
	~ x + y,
	data = df,
    subset = (z == "val"),
	method = "spearman",
	exact = FALSE
)
```

#### Effect size

The Spearman coefficient itself is the effect size.

| $\vert r_s\vert$ | effect size |
| :-: | :-: |
| $\approx 0.1$ | small |
| $\approx 0.3$ | medium |
| $\approx 0.5$ | large |

#### Power analysis

``` R
# Power analysis (NULL as required)
pwr::pwr.r.test(
	r = 0.998,
	n = 90,
	power = NULL,
	sig.level = 0.003
)
```

### 8.3 Kendall correlation

Kendall's correlation coefficient is based on **pairwise ordering**.

Use Kendall correlation when:

- the data are ordinal
- ties are common
- you want an interpretable pairwise ordering measure
- the sample size is not very large
- you want a more robust rank-based measure than Pearson

For any pair of observations $(i,j)$, with $i<j$, the pair is:

- **concordant** if the rankings of $X$ and $Y$ agree
- **discordant** if the rankings disagree

More precisely:

- concordant if $(X_i-X_j)(Y_i-Y_j) > 0$
- discordant if $(X_i-X_j)(Y_i-Y_j) < 0$

Let:

- $C$ = number of concordant pairs
- $D$ = number of discordant pairs

#### Kendall's tau without ties

If there are no ties,

$$
\tau = \frac{C-D}{\binom{n}{2}}.
$$

#### Kendall's tau-b with ties

With ties, the usual version is Kendall's $\tau_b$:

$$
\tau_b = \frac{C-D}{\sqrt{(C+D+T_X)(C+D+T_Y)}},
$$

where:

- $T_X$ adjusts for ties in $X$
- $T_Y$ adjusts for ties in $Y$

#### Null hypothesis

$$
H_0: \tau = 0
$$

This is a test of no ordinal association.

##### Implementation in R

``` R
# Kendall correlation
# H0: tau = 0
stats::cor.test(
  x = df$x,
  y = df$y,
  method = "kendall",
  alternative = "two.sided"
)
```
#### Effect size

Kendall's $\tau$ itself is the effect size.

| $\vert\tau\vert$ | effect size |
| :-: | :-: |
| $\approx 0.1$ | small |
| $\approx 0.3$ | medium |
| $\approx 0.5$ | large |

## 9 Regression

### 9.1 Linear regression

An indepth dive into linear regression in projects.

Assumptions
- Test homoskedasticity in the (fitted, resuals) space. The plot should show no relationship or a funnel pattern (heteroskedasticity).
- Test the normality of the residuals with a QQ plot, sample quantiles against theoretical quantiles.

Sample equation for an individual,

$$
y_i = a + \sum_{j=1}^p x_{i,p} + e_i
$$

#### Estimation

``` R
# Create a linear regression model
lmodel <- lm(y ~ x, data = df) 

# Results readout
summary(lmodel)

# Extract the fitted values and the residuals
fitted <- lmodel$fitted # fitted(lmodel)
resid <- lmodel$resid # resid(lmodel)
```

#### Interpolation

``` R
# Store the data in a data frame
x_df <- tibble(x = c(...)) # Interpolation input

# Interpolate with predict()
predict(lmodel, newdata = x_df)
```

#### Visualisation

``` R
# Plot the data
x_vec <- c(...) # Interpolation input

ggplot(df, aes(x = x, y = y, colour = z)) + 
  geom_point() +
  geom_smooth(method = "lm") +
  geom_vline(xintercept = x_vec) 
```

### 9.2 Logistic regression

An indepth dive into logistic regression in projects.

Sample equation for an individual,

$$
\ln{\left(\frac{\hat{p}}{1-\hat{p}}\right)} = a + \sum_{j=1}^p x_{i,p}
$$

#### Estimation

``` R
# Fit and estimate the model
logit_model <- glm(y ~ x, data = df, family = binomial)

# Results readout
summary(logit_model)
```

#### Likelihood ratio test

- LR test is analogous to the F-test in linear regression
- null deviance computed from the intercept-only model
- lower residual deviance indicates a better fit

$$\begin{align*}
G^2 &= D_0 - D_1 \sim \chi^2_{df_0 - df_1} \\
D_0 &= -2\ln \mathcal{L}(\hat{p}_0 \mid y) \\
D_1 &= -2\ln \mathcal{L}(\hat{p}_i \mid y)
\end{align*}$$

``` R
# Likelihood ratio test
# H0: all slope coefficients = 0
chs_val <- logit_model$null.deviance - logit_model$deviance
df_val <- logit_model$df.null - logit_model$df.residual

pchisq(q = chs_val, df = df_val, lower.tail = FALSE)
```

#### Interpolation

``` R
# Interpolate
x_df <- tibble(x = c(...)) # Interpolation input

predict(logit_model, x_df, type = "response")
```

#### Visualisation

``` R
# Plot the data
x_vec <- c(...) # Interpolation input

ggplot(df, aes(x = x, y = y)) + 
  geom_point() +
  geom_smooth(method = "glm",
             method.args = list("binomial")) +
  geom_vline(xintercept = x_vec) 
```